# 📊 Урок 11 — Метрики и confusion matrix (материалы преподавателя)

Демонстрационный ноутбук. Каждый блок: **что делаем · зачем · что ожидаем** → код → разбор вывода.

> 🎯 Цель: показать, что accuracy недостаточно, и научить читать матрицу ошибок, precision, recall, F1.

## Блок 1 · Модель в Pipeline (с урока 10)
**Что делаем:** обучаем лес в Pipeline на Titanic и получаем предсказания.
**Зачем:** дальше будем разбирать её ошибки.

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

num = ['age','fare','sibsp','parch']; cat = ['sex','pclass','embarked']
prep = ColumnTransformer([
    ('num', Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]), num),
    ('cat', Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]), cat),
])

df = sns.load_dataset('titanic')
X = df[num+cat]; y = df['survived']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = Pipeline([('prep',prep),('forest',RandomForestClassifier(n_estimators=100,random_state=42))])
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print('Модель обучена, предсказания готовы.')

## Блок 2 · Провокация «99% и бесполезно»
**Что делаем:** «тупой» baseline, который всегда предсказывает большинство.
**Что ожидаем:** accuracy приличная, но recall = 0 — модель никого не поймала.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
dp = dummy.predict(X_test)
print(f'Тупой baseline: accuracy={accuracy_score(y_test,dp):.0%}, recall={recall_score(y_test,dp):.0%}')
print('Вывод: accuracy может выглядеть прилично при полностью бесполезной модели.')

## Блок 3 · Матрица ошибок
**Что делаем:** строим confusion matrix на реальных предсказаниях.
**Что ожидаем:** диагональ — верные ответы, вне диагонали — два вида ошибок.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=['погиб','выжил'], cmap='Purples')
plt.title('Матрица ошибок'); plt.show()

## Блок 4 · precision, recall, F1
**Что делаем:** полный отчёт по метрикам.
**Что ожидаем:** увидим разницу между precision и recall для класса «выжил».

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=['погиб','выжил']))

## Блок 5 (эксперимент) · Порог решения
**Что делаем:** меняем порог и смотрим, как precision и recall перетягивают друг друга.
**Что ожидаем:** ниже порог → выше recall, ниже precision.

In [ ]:
from sklearn.metrics import precision_score, recall_score
proba = model.predict_proba(X_test)[:,1]
for thr in [0.3, 0.5, 0.7]:
    pred = (proba >= thr).astype(int)
    print(f'порог {thr}:  precision={precision_score(y_test,pred):.0%}  recall={recall_score(y_test,pred):.0%}')

---
**Итог урока.** Accuracy врёт на несбалансированных данных. Смотрим матрицу ошибок и выбираем метрику по цене ошибки: precision (против ложных тревог) или recall (против пропусков).